# Brain Tumor MRI Classification

**Task:** Multi-class classification of brain MRI scans
- Glioma
- Meningioma
- Pituitary
- Normal

**Model:** EfficientNet-B0

**Dataset:** [Brain Tumor MRI Dataset (Kaggle)](https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri)

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '../..')  # Add project root

import torch
from shared.config import DEVICE, BATCH_SIZE, EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE
from shared.models import create_model
from shared.pipelines.dataset import create_dataloaders
from shared.pipelines.train import train_model

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# ── Load Configuration ─────────────────────────────────────
from config import PROJECT_ID, MODEL_NAME, CLASSES, IMG_SIZE
from shared.config import get_trained_model_path

print(f"Project: {PROJECT_ID}")
print(f"Model: {MODEL_NAME}")
print(f"Classes: {CLASSES}")
print(f"Image Size: {IMG_SIZE}")

In [ ]:
# ── Load Dataset ───────────────────────────────────────────
data_root = "data"
train_loader, val_loader, test_loader, dataset_classes = create_dataloaders(
    data_root=data_root,
    class_names=CLASSES,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=2,
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")

In [ ]:
# ── Create Model ───────────────────────────────────────────
model = create_model(MODEL_NAME, num_classes=len(CLASSES))
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── Train Model ────────────────────────────────────────────
checkpoint_path = str(get_trained_model_path(PROJECT_ID))

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_path=checkpoint_path,
)

print("\n✓ Training complete!")

In [ ]:
# ── Evaluate on Test Set ───────────────────────────────────
from shared.utils.metrics import compute_metrics
from shared.utils.visualization import plot_confusion_matrix
from IPython.display import display
import matplotlib.pyplot as plt

# Load best model
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

metrics = compute_metrics(all_labels, all_preds, all_probs)

print(f"Test Accuracy:  {metrics['accuracy']:.4f}")
print(f"Test Precision: {metrics['precision']:.4f}")
print(f"Test Recall:    {metrics['recall']:.4f}")
print(f"Test F1-Score:  {metrics['f1_score']:.4f}")
if metrics['roc_auc']:
    print(f"Test ROC-AUC:   {metrics['roc_auc']:.4f}")

# Display confusion matrix
cm_b64 = plot_confusion_matrix(metrics['confusion_matrix'], CLASSES)
from IPython.display import Image as IPImage
import base64
display(IPImage(base64.b64decode(cm_b64)))

In [ ]:
# ── Visualise with XAI ─────────────────────────────────────
from shared.explainers.xai_factory import run_all_explainers
from shared.pipelines.transforms import get_inference_transform
from PIL import Image
import numpy as np

# Load a test image
import os, random
test_dir = "data/test"
class_dir = random.choice(os.listdir(test_dir))
img_path = random.choice(os.listdir(os.path.join(test_dir, class_dir)))
img_full_path = os.path.join(test_dir, class_dir, img_path)
print(f"Sample: {img_full_path}")

pil_image = Image.open(img_full_path).convert('RGB')
transform = get_inference_transform(IMG_SIZE)
input_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)

# Run all explainers
result = run_all_explainers(
    model=model,
    image_tensor=input_tensor,
    class_names=CLASSES,
    device=DEVICE,
)

print(f"\nPrediction: {result['predictions']['predicted_class']}")
print(f"Confidence: {result['predictions']['confidence']:.2%}")

# Display each explainer
from IPython.display import display, HTML
import base64

html = '<div style="display:grid; grid-template-columns:repeat(2,1fr); gap:10px;">'
for key, exp in result['explanations'].items():
    if 'overlay_base64' in exp:
        html += f'''
            <div style="text-align:center; border:1px solid #ddd; border-radius:8px; padding:10px;">
                <h4>{exp['label']}</h4>
                <img src="data:image/png;base64,{exp['overlay_base64']}" style="width:100%; border-radius:4px;"/>
                <p style="font-size:0.8rem; color:#666;">{exp['description']}</p>
            </div>'
html += '</div>'
display(HTML(html))

In [ ]:
# ── Training History Plot ──────────────────────────────────
import matplotlib.pyplot as plt

train_acc = [m['accuracy'] for m in history['train']]
val_acc = [m['accuracy'] for m in history['val']]
train_loss = [m['loss'] for m in history['train']]
val_loss = [m['loss'] for m in history['val']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_acc, label='Train Acc', linewidth=2)
ax1.plot(val_acc, label='Val Acc', linewidth=2)
ax1.set_title('Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_loss, label='Train Loss', linewidth=2)
ax2.plot(val_loss, label='Val Loss', linewidth=2)
ax2.set_title('Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best Validation Accuracy: {max(val_acc):.4f}")

---
**Next Steps:**
1. Model weights saved to `trained_models/01_brain_tumor_best.pth`
2. Run `python run.py` from project root to start the web interface
3. Upload a brain MRI scan and see predictions with XAI heatmaps